# Multi-Class CNN Lab (PyTorch)

In this lab, you will build and compare several CNN approaches for an **8-class clothing classification task**.

You will:
- inspect the dataset
- build a baseline CNN from scratch
- evaluate the model on the validation set
- reduce overfitting with data augmentation
- apply transfer learning
- fine-tune a pre-trained model
- predict classes for **unlabeled test images**

> **This is the exercise version.** You already built and trained a CNN in Lab 1 (Cats vs Dogs). Several code cells below are left for **you to complete** (marked `# TODO`), reusing and adapting what you wrote in Lab 1. The surrounding infrastructure (data loading, plotting, metrics) is provided so you can focus on the deep learning logic.

# Learning Objectives

By the end of this lab, you should be able to:

- explain why CNNs are effective for image classification
- train a multiclass CNN from scratch
- interpret learning curves and detect overfitting
- evaluate a model using confusion matrices and classification reports **when labels are available**
- use data augmentation appropriately
- apply transfer learning with a pre-trained model
- fine-tune part of a pre-trained model for better performance
- predict classes for previously unseen **unlabeled** images

# Dataset Structure

Download the image dataset "data_clothes.zip" available here: https://drive.google.com/file/d/1M1sD4DKg1_7zk6xgAIMbPG9y2pPYPc7R/view?usp=sharing.
This dataset contains 8 categories of images distributed as shown in the following structure (in total 22 directories):

```text
data/
├── train/
│   ├── accessories/
│   ├── jackets/
│   ├── jeans/
│   ├── knitwear/
│   ├── shirts/
│   ├── shoes/
│   ├── shorts/
│   └── tees/
├── validation/
│   ├── accessories/
│   ├── jackets/
│   ├── jeans/
│   ├── knitwear/
│   ├── shirts/
│   ├── shoes/
│   ├── shorts/
│   └── tees/
└── test/
    └── unknown/
        ├── image1.jpg
        ├── image2.jpg
        └── ...
```

In [ ]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from torchvision import datasets, transforms, models
from torchvision.models import MobileNet_V2_Weights
from PIL import Image

from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

print("PyTorch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

# Part 1 — Load and Inspect the Dataset

We start by checking that the directories exist and that the dataset is structured correctly.

In [ ]:
# Running locally: no Google Drive needed.
# Download data_clothes.zip from the course Drive and put it next to this notebook.

In [ ]:
import os
from zipfile import ZipFile
file_name = "data_clothes.zip"

if os.path.isdir("data_clothes"):
    print("data_clothes/ already extracted")
else:
    with ZipFile(file_name, 'r') as zip_ref:
        zip_ref.extractall("data_clothes")
    print('Done')

In [ ]:
base_dir = "data_clothes"

train_dir = os.path.join(base_dir, "train")
validation_dir = os.path.join(base_dir, "valid")
unknown_test_dir = os.path.join(base_dir, "test", "unknown")

print("Train exists:", os.path.exists(train_dir))
print("Validation exists:", os.path.exists(validation_dir))
print("Unknown test exists:", os.path.exists(unknown_test_dir))

In [ ]:
class_names = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
class_names

In [ ]:
print("Number of classes:", len(class_names))
print("Classes:", class_names)

In [ ]:
def count_images_in_directory(directory):
    counts = {}
    total = 0
    for class_name in sorted(os.listdir(directory)):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            n = len([f for f in os.listdir(class_path) if not f.startswith(".")])
            counts[class_name] = n
            total += n
    return counts, total

train_counts, train_total = count_images_in_directory(train_dir)
val_counts, val_total = count_images_in_directory(validation_dir)

print("Train total:", train_total)
print(train_counts)
print()
print("Validation total:", val_total)
print(val_counts)

unknown_files = sorted([f for f in os.listdir(unknown_test_dir) if not f.startswith(".")])
print()
print("Unknown test images:", len(unknown_files))
print(unknown_files[:10])

# Create Datasets and DataLoaders

For the baseline experiment, we normalize pixel values using `ToTensor()` (equivalent to `rescale=1./255`).

**Exercise:** In Lab 2 you built `train_generator` / `validation_generator` with `datasets.ImageFolder` + `DataLoader`. Do the same here.

In [ ]:
img_height = 150
img_width = 150
batch_size = 32

basic_transform = transforms.Compose([
    transforms.Resize((img_height, img_width)),
    transforms.ToTensor(),
])

# TODO: build train_dataset and validation_dataset with datasets.ImageFolder(..., transform=basic_transform)
# using train_dir and validation_dir

train_dataset = ...
validation_dataset = ...

# TODO: wrap them in DataLoader objects called train_generator and validation_generator
# (shuffle=True for training, shuffle=False for validation), same as in Lab 2

train_generator = ...
validation_generator = ...

num_classes = len(train_dataset.classes)

In [ ]:
print("Class indices:", train_dataset.class_to_idx)

In [ ]:
images, labels = next(iter(train_generator))

plt.figure(figsize=(12, 8))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i].permute(1, 2, 0).numpy())
    plt.title(class_names[labels[i].item()])
    plt.axis("off")
plt.tight_layout()
plt.show()

# Part 2 — Baseline CNN From Scratch

We now build a first CNN from scratch.  
This model serves as a baseline for all later comparisons.

**Exercise:** Build the same style of CNN as in Lab 3 (`BinaryCNN`), but adapted for **8-class** classification:

- convolution blocks: `Conv2d(3,32,3) → ReLU → MaxPool2d(2)`, then `Conv2d(32,64,3) → ReLU → MaxPool2d(2)`, then `Conv2d(64,128,3) → ReLU → MaxPool2d(2)`
- classifier head: `Flatten → Linear(128*17*17, 256) → ReLU → Dropout(0.4) → Linear(256, num_classes)`

**Key difference from Lab 1:** the last layer now has `num_classes` (8) outputs instead of 1, because this is multi-class, not binary. The output stays a set of **raw logits** — no softmax inside the model (we'll use `CrossEntropyLoss`, which applies it internally).

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # TODO: build self.features (3 conv+relu+maxpool blocks, same shapes as Lab 2's BinaryCNN)
        self.features = nn.Sequential(
            # YOUR CODE HERE
        )

        # TODO: build self.classifier
        # Flatten -> Linear(128*17*17, 256) -> ReLU -> Dropout(0.4) -> Linear(256, num_classes)
        self.classifier = nn.Sequential(
            # YOUR CODE HERE
        )

    def forward(self, x):
        # TODO: pass x through self.features then self.classifier and return the result
        pass

baseline_model = BaselineCNN(num_classes).to(device)

In [ ]:
total_params = sum(p.numel() for p in baseline_model.parameters())
print(baseline_model)
print(f"Total parameters: {total_params:,}")

**Exercise:** define the loss and optimizer.

- In Lab 1 (binary) you used `nn.BCEWithLogitsLoss()`. Here, since we have 8 classes, use the loss you already used for multi-class problems.
- Keep `Adam` as optimizer with `lr=1e-4`, same as the reference architecture.

In [ ]:
# TODO: define criterion (multi-class loss) and optimizer (Adam, lr=1e-4)
criterion = ...
optimizer = ...

**Exercise — new concept:** `ReduceLROnPlateau` scheduler.

Unlike Lab 1, we now also want to **automatically reduce the learning rate** when validation loss stops improving, mirroring Keras's `ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)`.

PyTorch provides `torch.optim.lr_scheduler.ReduceLROnPlateau`. Look up its signature and create a `scheduler` that:
- watches a minimized quantity (`mode="min"`)
- multiplies the learning rate by `0.5` when there's no improvement (`factor=0.5`)
- waits `3` epochs without improvement before reducing (`patience=3`)

We will also implement **early stopping** manually (patience `5` epochs, monitoring validation loss, restoring the best weights) — like Keras's `EarlyStopping(patience=5, restore_best_weights=True)`.

In [ ]:
# TODO: create the ReduceLROnPlateau scheduler wrapping `optimizer`
scheduler = ...

early_stopping_patience = 5

**Exercise:** write `evaluate_multiclass`, adapting the `evaluate_binary` function you wrote in Lab 2.

Differences from the binary version:
- labels stay as `long` integers (no `.float().unsqueeze(1)`)
- the loss is your multi-class `criterion` (raw logits vs. integer labels)
- predictions come from `torch.argmax(logits, dim=1)` instead of `sigmoid(...) > 0.5`

In [ ]:
def evaluate_multiclass(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in data_loader:
            # TODO: move xb, yb to device
            # TODO: compute logits = model(xb)
            # TODO: compute loss = criterion(logits, yb)
            # TODO: accumulate total_loss, correct, total (see Lab 2's evaluate_binary for the pattern)
            pass
    return total_loss / total, correct / total

**Exercise — the big one:** write `train_with_early_stopping`.

Start from the `train_binary_model` function you used in Lab 1, and adapt/extend it:

1. Same training loop structure: `model.train()`, iterate over batches, `zero_grad()` / `backward()` / `step()`, track `running_loss`, `correct`, `total`.
2. Use `torch.argmax(logits, dim=1)` for predictions (multi-class), and integer labels directly.
3. At the end of each epoch, call `scheduler.step(val_loss)` to let `ReduceLROnPlateau` adjust the learning rate.
4. Implement early stopping manually:
   - keep track of `best_val_loss` (start at `float("inf")`) and `best_state` (a `copy.deepcopy(model.state_dict())`)
   - if `val_loss` improves, save the new best state and reset a counter
   - if it doesn't improve for `patience` epochs in a row, print a message and `break` out of the loop
5. After the loop, reload the best weights with `model.load_state_dict(best_state)` — this is the equivalent of Keras's `restore_best_weights=True`.

Return the same kind of `history` dictionary as in Lab 1 (`"accuracy"`, `"val_accuracy"`, `"loss"`, `"val_loss"`).

In [ ]:
def train_with_early_stopping(model, train_loader, val_loader, criterion, optimizer, scheduler,
                               device, epochs=20, patience=5):
    history = {"accuracy": [], "val_accuracy": [], "loss": [], "val_loss": []}
    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for xb, yb in train_loader:
            # TODO: training step (move to device, zero_grad, forward, loss, backward, step)
            # TODO: accumulate running_loss, correct, total
            pass

        train_loss = running_loss / total
        train_acc = correct / total

        # TODO: compute val_loss, val_acc using evaluate_multiclass

        history["loss"].append(train_loss)
        history["accuracy"].append(train_acc)
        # TODO: append val_loss and val_acc to history

        print(f"Epoch {epoch+1}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f}")
        # TODO: also print val_loss and val_accuracy in the line above

        # TODO: call scheduler.step(...) with the validation loss

        # TODO: early stopping logic:
        #   if val_loss improved -> save best_state, reset counter
        #   else -> increment counter; if counter >= patience, print message and break

    # TODO: restore best_state into the model before returning

    return history

In [ ]:
history_baseline = train_with_early_stopping(
    baseline_model, train_generator, validation_generator,
    criterion, optimizer, scheduler, device, epochs=20, patience=5
)

# Analyze the Learning Curves

Look at both **accuracy** and **loss** on training and validation sets.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_baseline["accuracy"], label="Training Accuracy")
plt.plot(history_baseline["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Baseline CNN - Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_baseline["loss"], label="Training Loss")
plt.plot(history_baseline["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Baseline CNN - Loss")
plt.legend()
plt.show()

# Part 3 — Evaluation on the Validation Set

Because the `test/unknown` folder has **no labels**, we use the **validation set** for detailed evaluation:
- loss
- accuracy
- confusion matrix
- classification report
- error analysis

The unknown test images will be used later only for **prediction / inference**.

In [ ]:
val_loss_baseline, val_acc_baseline = evaluate_multiclass(baseline_model, validation_generator, criterion, device)

print("Baseline Validation Loss:", val_loss_baseline)
print("Baseline Validation Accuracy:", val_acc_baseline)

**Exercise:** gather predictions and true labels over the whole validation set, like you did for the test set in Lab 1. Remember: this is multi-class, so use `torch.softmax(logits, dim=1)` and `np.argmax(..., axis=1)` instead of `sigmoid` + threshold.

In [ ]:
baseline_model.eval()
all_probs = []
all_true = []

with torch.no_grad():
    for xb, yb in validation_generator:
        # TODO: move xb to device, compute logits, then softmax probabilities
        # TODO: append probs (as numpy) to all_probs, and yb (as numpy) to all_true
        pass

y_pred_probs_baseline = np.concatenate(all_probs)
y_pred_baseline = np.argmax(y_pred_probs_baseline, axis=1)
y_true_val = np.concatenate(all_true)

ordered_class_names = [name for name, idx in sorted(validation_dataset.class_to_idx.items(), key=lambda x: x[1])]

print("Number of validation samples:", len(y_true_val))
print("Ordered class names:", ordered_class_names)

In [ ]:
cm_baseline = confusion_matrix(
    y_true_val,
    y_pred_baseline,
    labels=list(range(len(ordered_class_names)))
)

fig, ax = plt.subplots(figsize=(8, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_baseline, display_labels=ordered_class_names)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
plt.title("Baseline CNN - Validation Confusion Matrix")
plt.show()

In [ ]:
print(classification_report(
    y_true_val,
    y_pred_baseline,
    labels=list(range(len(ordered_class_names))),
    target_names=ordered_class_names
))

# Visualize Validation Predictions

Let us inspect a few correct and incorrect predictions on the validation set.

In [ ]:
sample_images, sample_labels = next(iter(validation_generator))
with torch.no_grad():
    sample_probs = torch.softmax(baseline_model(sample_images.to(device)), dim=1).cpu().numpy()
sample_preds = np.argmax(sample_probs, axis=1)

plt.figure(figsize=(14, 8))
for i in range(min(8, len(sample_images))):
    plt.subplot(2, 4, i + 1)
    plt.imshow(sample_images[i].permute(1, 2, 0).numpy())
    true_label = ordered_class_names[sample_labels[i].item()]
    pred_label = ordered_class_names[sample_preds[i]]
    plt.title(f"True: {true_label}\nPred: {pred_label}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
all_val_images = []
for xb, yb in validation_generator:
    all_val_images.append(xb.numpy())
all_val_images = np.vstack(all_val_images)

misclassified_idx = np.where(y_pred_baseline != y_true_val)[0]

plt.figure(figsize=(14, 8))
for i, idx in enumerate(misclassified_idx[:8]):
    plt.subplot(2, 4, i + 1)
    plt.imshow(np.transpose(all_val_images[idx], (1, 2, 0)))
    true_label = ordered_class_names[y_true_val[idx]]
    pred_label = ordered_class_names[y_pred_baseline[idx]]
    plt.title(f"True: {true_label}\nPred: {pred_label}")
    plt.axis("off")
plt.tight_layout()
plt.show()

# Part 4 — Improve the Baseline with Data Augmentation

Data augmentation helps the model generalize better by showing slightly modified versions of training images.

**Exercise:** build `train_transform_aug`, a `transforms.Compose` that adds random augmentations on top of the basic resize/tensor pipeline:

- `transforms.RandomRotation(20)`
- `transforms.RandomAffine(degrees=0, translate=(0.1, 0.1))`
- `transforms.RandomResizedCrop((img_height, img_width), scale=(0.9, 1.1))`
- `transforms.RandomHorizontalFlip()`

Keep `validation_transform_aug` **without** any augmentation (only resize + `ToTensor()`) — we never augment validation or test data.

In [ ]:
# TODO: build train_transform_aug with the augmentations listed above (+ Resize and ToTensor)
train_transform_aug = transforms.Compose([
    # YOUR CODE HERE
])

# TODO: build validation_transform_aug (Resize + ToTensor only, no augmentation)
validation_transform_aug = transforms.Compose([
    # YOUR CODE HERE
])

In [ ]:
train_dataset_aug = datasets.ImageFolder(train_dir, transform=train_transform_aug)
validation_dataset_aug = datasets.ImageFolder(validation_dir, transform=validation_transform_aug)

train_generator_aug = DataLoader(train_dataset_aug, batch_size=batch_size, shuffle=True)
validation_generator_aug = DataLoader(validation_dataset_aug, batch_size=batch_size, shuffle=False)

**Exercise:** create a fresh model, criterion, optimizer, and scheduler for this experiment (don't reuse the already-trained `baseline_model` — we want a model trained from scratch on augmented data, for a fair comparison).

In [ ]:
# TODO: create augmented_model = BaselineCNN(num_classes).to(device)
# TODO: create criterion_aug, optimizer_aug (Adam, lr=1e-4), scheduler_aug (ReduceLROnPlateau, same settings as before)
augmented_model = ...
criterion_aug = ...
optimizer_aug = ...
scheduler_aug = ...

In [ ]:
history_augmented = train_with_early_stopping(
    augmented_model, train_generator_aug, validation_generator_aug,
    criterion_aug, optimizer_aug, scheduler_aug, device, epochs=20, patience=5
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_baseline["val_accuracy"], label="Baseline Val Accuracy")
plt.plot(history_augmented["val_accuracy"], label="Augmented Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Baseline vs Data Augmentation")
plt.legend()
plt.show()

In [ ]:
val_loss_aug, val_acc_aug = evaluate_multiclass(augmented_model, validation_generator_aug, criterion_aug, device)
print("Augmented Model Validation Loss:", val_loss_aug)
print("Augmented Model Validation Accuracy:", val_acc_aug)

# Part 5 — Transfer Learning with MobileNetV2

Instead of learning everything from scratch, we can reuse features learned on ImageNet.
This often works well when the dataset is not huge.

> Note: torchvision's pretrained MobileNetV2 expects the standard ImageNet normalization (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`) rather than Keras's `preprocess_input`.

In [ ]:
img_height_tl = 224
img_width_tl = 224

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform_tl = transforms.Compose([
    transforms.Resize((img_height_tl, img_width_tl)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

validation_transform_tl = transforms.Compose([
    transforms.Resize((img_height_tl, img_width_tl)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

In [ ]:
train_dataset_tl = datasets.ImageFolder(train_dir, transform=train_transform_tl)
validation_dataset_tl = datasets.ImageFolder(validation_dir, transform=validation_transform_tl)

train_generator_tl = DataLoader(train_dataset_tl, batch_size=batch_size, shuffle=True)
validation_generator_tl = DataLoader(validation_dataset_tl, batch_size=batch_size, shuffle=False)

**Exercise:** build the transfer learning model.

1. Load the pretrained backbone: `models.mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)`, and keep only its `.features` (the convolutional part, without the original classifier).
2. **Freeze** all of its parameters (`param.requires_grad = False`) — we don't want to update ImageNet-learned weights yet.
3. Complete the `TransferModel` class:
   - `self.pool`: a `nn.AdaptiveAvgPool2d(1)` (equivalent to Keras's `GlobalAveragePooling2D`)
   - `self.classifier`: `Flatten → Dropout(0.3) → Linear(1280, 128) → ReLU → Dropout(0.3) → Linear(128, num_classes)`
     (`1280` is the number of output channels of MobileNetV2's feature extractor)
   - `forward`: pass `x` through `self.base_model`, then `self.pool`, then `self.classifier`

In [ ]:
# load the pretrained MobileNetV2 and keep only its `.features`
base_model = models.mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
base_model = base_model.features  # convolutional feature extractor only (no top classifier)

# freeze every parameter in base_model
for param in base_model.parameters():
    param.requires_grad = False

class TransferModel(nn.Module):
    def __init__(self, base_model, num_classes):
        super().__init__()
        self.base_model = base_model
        self.pool = nn.AdaptiveAvgPool2d(1)  # GlobalAveragePooling2D equivalent
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(1280, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.base_model(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

transfer_model = TransferModel(base_model, num_classes).to(device)

In [ ]:
total_params = sum(p.numel() for p in transfer_model.parameters())
trainable_params = sum(p.numel() for p in transfer_model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
# Sanity check: trainable_params should be much smaller than total_params, since the backbone is frozen

In [ ]:
# TODO: criterion_tl, optimizer_tl (Adam, lr=1e-4), scheduler_tl (ReduceLROnPlateau, same settings)
criterion_tl = ...
optimizer_tl = ...
scheduler_tl = ...

In [ ]:
history_transfer = train_with_early_stopping(
    transfer_model, train_generator_tl, validation_generator_tl,
    criterion_tl, optimizer_tl, scheduler_tl, device, epochs=15, patience=5
)

In [ ]:
val_loss_transfer, val_acc_transfer = evaluate_multiclass(transfer_model, validation_generator_tl, criterion_tl, device)
print("Transfer Learning Validation Loss:", val_loss_transfer)
print("Transfer Learning Validation Accuracy:", val_acc_transfer)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_baseline["val_accuracy"], label="Baseline")
plt.plot(history_augmented["val_accuracy"], label="Augmented")
plt.plot(history_transfer["val_accuracy"], label="Transfer Learning")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy Comparison")
plt.legend()
plt.show()

# Part 6 — Fine-Tuning the Pre-Trained Model

Once the classifier on top is trained, we can unfreeze a few higher layers of the pre-trained model and continue training with a very small learning rate.

> Note: torchvision's MobileNetV2 `features` module is organized as a sequence of building blocks, so "unfreeze the last 30 layers" is adapted here to "unfreeze the last few blocks", which plays the same role.

**Exercise:**

1. Unfreeze **all** parameters of `transfer_model.base_model` first.
2. Then re-freeze all blocks **except the last 4** (`list(transfer_model.base_model.children())` gives you the blocks in order).
3. Create a new optimizer with a much smaller learning rate (`1e-5`), that only updates parameters where `requires_grad=True`. Hint: `filter(lambda p: p.requires_grad, transfer_model.parameters())`.
4. Create a new `ReduceLROnPlateau` scheduler wrapping this optimizer.

In [ ]:
# TODO: unfreeze all parameters in transfer_model.base_model

# TODO: get the list of blocks with list(transfer_model.base_model.children())
# then re-freeze all of them except the last 4

blocks = list(transfer_model.base_model.children())
for block in blocks[:-4]:
    for param in block.parameters():
        param.requires_grad = False

trainable_blocks = sum(1 for block in blocks[-4:] for _ in block.parameters())
print("Number of trainable parameter tensors in the last blocks:", trainable_blocks)

In [ ]:
# TODO: optimizer_ft (Adam, lr=1e-5, only parameters with requires_grad=True)
# TODO: scheduler_ft (ReduceLROnPlateau, same settings as before)
# TODO: criterion_ft (same loss as before)
optimizer_ft = ...
scheduler_ft = ...
criterion_ft = ...

In [ ]:
history_finetune = train_with_early_stopping(
    transfer_model, train_generator_tl, validation_generator_tl,
    criterion_ft, optimizer_ft, scheduler_ft, device, epochs=10, patience=5
)

In [ ]:
val_loss_finetune, val_acc_finetune = evaluate_multiclass(transfer_model, validation_generator_tl, criterion_ft, device)
print("Fine-Tuned Model Validation Loss:", val_loss_finetune)
print("Fine-Tuned Model Validation Accuracy:", val_acc_finetune)

# Part 7 — Predict Classes for Unlabeled Test Images

Now we use the trained model to predict the classes of the images stored in `data/test/unknown`.

Because these images have **no labels**, this is **inference only**:
- no accuracy
- no confusion matrix
- no classification report

In [ ]:
best_model = transfer_model  # change this if another model performed better
best_model_name = "Fine-tuned Transfer Model"  # update if needed
use_transfer_preprocessing = True  # False for baseline/augmented, True for transfer/fine-tuned

print("Selected model for unknown predictions:", best_model_name)

In [ ]:
unknown_image_files = sorted([
    f for f in os.listdir(unknown_test_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
])

print("Unknown images found:", len(unknown_image_files))
print(unknown_image_files)

**Exercise:** build the preprocessing pipeline for the unknown images (pick the right transform depending on `use_transfer_preprocessing`), load each image with `PIL.Image.open(...).convert("RGB")`, apply the transform, and stack everything into one tensor with `torch.stack(...)`.

In [ ]:
if use_transfer_preprocessing:
    target_h, target_w = img_height_tl, img_width_tl
    predict_transform = transforms.Compose([
        transforms.Resize((target_h, target_w)),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ])
else:
    target_h, target_w = img_height, img_width
    predict_transform = transforms.Compose([
        transforms.Resize((target_h, target_w)),
        transforms.ToTensor(),
    ])

unknown_tensors = []
unknown_filenames = []

for fname in unknown_image_files:
    # TODO: open the image at os.path.join(unknown_test_dir, fname), convert to RGB,
    # apply predict_transform, and append the result to unknown_tensors
    # (also append fname to unknown_filenames)
    pass

# TODO: stack unknown_tensors into a single tensor called unknown_images
unknown_images = ...

print("Unknown images tensor shape:", unknown_images.shape)

In [ ]:
best_model.eval()
with torch.no_grad():
    logits_unknown = best_model(unknown_images.to(device))
    pred_probs_unknown = torch.softmax(logits_unknown, dim=1).cpu().numpy()

pred_classes_unknown = np.argmax(pred_probs_unknown, axis=1)
pred_labels_unknown = [ordered_class_names[i] for i in pred_classes_unknown]
pred_confidence_unknown = np.max(pred_probs_unknown, axis=1)

for fname, label, conf in zip(unknown_filenames, pred_labels_unknown, pred_confidence_unknown):
    print(f"{fname} -> {label} ({conf:.2f})")

In [ ]:
plt.figure(figsize=(14, 8))
for i in range(min(8, len(unknown_images))):
    plt.subplot(2, 4, i + 1)
    display_img = Image.open(os.path.join(unknown_test_dir, unknown_filenames[i])).convert("RGB").resize((target_h, target_w))
    plt.imshow(display_img)
    plt.title(f"{pred_labels_unknown[i]}\n{pred_confidence_unknown[i]:.2f}")
    plt.axis("off")

plt.tight_layout()
plt.show()

# Part 8 — Model Comparison Summary

Summarize the performance of:
- baseline CNN
- CNN with augmentation
- transfer learning
- fine-tuned transfer learning

Use the **validation set** for comparison, since the unknown test folder has no labels.

In [ ]:
results = {
    "Baseline CNN": val_acc_baseline,
    "CNN + Augmentation": val_acc_aug,
    "Transfer Learning": val_acc_transfer,
    "Fine-Tuned Transfer": val_acc_finetune,
}

for model_name, acc in results.items():
    print(f"{model_name}: {acc:.4f}")

# Experimental Extensions

Choose **one or two** of the following extensions:

- increase image size
- try another backbone such as EfficientNetB0 (`torchvision.models.efficientnet_b0`)
- add stronger augmentation
- compare Adam vs RMSprop
- compare `Dropout(0.3)` vs `Dropout(0.5)`
- compute top-2 predictions for unknown images
- train longer with early stopping

# Final Questions

Answer these global questions at the end of the lab:

1. What was the purpose of each major stage of the notebook?
   - baseline CNN
   - validation evaluation
   - data augmentation
   - transfer learning
   - fine-tuning
   - prediction on unknown images

2. Which model performed best on the validation set, and why do you think it worked better?

3. What signs of overfitting or underfitting did you observe during training?

4. Did data augmentation help in your case? Explain your answer using the validation curves.

5. How did transfer learning compare with training a CNN from scratch?

6. Did fine-tuning improve the transfer learning model? Why might that happen?

7. Which classes were easiest and hardest to classify on the validation set?

8. What kinds of mistakes did the model make, and were those mistakes understandable visually?

9. Why can we not compute accuracy on the `test/unknown` folder?

10. If you had one more hour to improve the notebook results, what would you try next?

11. What was harder to write yourself this time compared to Lab 1: the model architecture, the training loop with early stopping, or the transfer learning setup? Why?